In [ ]:
import numpy as np
import pandas as pd
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path


In [ ]:
from pathlib import Path
import pandas as pd

# path to the project folder
base_path = Path.cwd().resolve()  # this notebook's own folder (src/notebooks), where the CSVs live

# load files (directly in the folder)
all_mps = pd.read_csv(base_path / "all_mps_orgid-1.csv")
whole_eeg = pd.read_csv(base_path / "whole_eeg_sums-1.csv")

# convert time columns
all_mps["time"] = pd.to_datetime(all_mps["time"], utc=True)
whole_eeg["time"] = pd.to_datetime(whole_eeg["time"], utc=True)

In [ ]:
def chapter1_kpis(all_mps, whole_eeg, org_id, tol=0.01):
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    mps = all_mps.copy()
    eeg = whole_eeg.copy()

    mps["time"] = pd.to_datetime(mps["time"], utc=True)
    eeg["time"] = pd.to_datetime(eeg["time"], utc=True)

    mps = mps[mps["org_id"] == org_id]
    eeg = eeg[eeg["org_id"] == org_id]

    mps["day"] = mps["time"].dt.date
    eeg["day"] = eeg["time"].dt.date

    exp = 96
    comp = mps.groupby(["day", "mp_id"])["time"].nunique().reset_index(name="n")
    comp["completeness"] = (comp["n"] / exp).clip(0, 1)

    valid_row = (
        (mps["wt_meas_cons"].isna() | (mps["wt_meas_cons"] >= 0)) &
        (mps["wt_meas_gen"].isna()  | (mps["wt_meas_gen"]  >= 0)) &
        (mps["comm_pot"].isna()     | (mps["comm_pot"]     >= 0)) &
        (mps["comm_cov"].isna()     | (mps["comm_cov"]     >= 0)) &
        (mps["period_interval"].astype(str) == "0 days 00:15:00")
    )
    mps["valid_row"] = valid_row
    val = mps.groupby(["day","mp_id"])["valid_row"].mean().reset_index(name="validity")

    mps = mps.sort_values(["mp_id","time"])
    dcons = mps.groupby(["day","mp_id"])["wt_meas_cons"].diff().abs()
    thr = dcons.groupby([mps["day"], mps["mp_id"]]).transform(lambda x: max(1e-9, 10*np.nanmedian(x)))
    plaus = (dcons.isna() | (dcons <= thr)).groupby([mps["day"], mps["mp_id"]]).mean().reset_index(name="plausibility")

    mp_day = comp.merge(val, on=["day","mp_id"]).merge(plaus, on=["day","mp_id"])

    agg = mps.groupby("time", as_index=False)["wt_meas_cons"].sum().rename(columns={"wt_meas_cons": "sum_mps"})
    merged = agg.merge(eeg[["time", "sum_wt_meas_cons"]], on="time", how="inner").sort_values("time")
    merged["day"] = merged["time"].dt.date
    merged["abs_diff"] = (merged["sum_mps"] - merged["sum_wt_meas_cons"]).abs()
    merged["abs_ref"] = merged["sum_wt_meas_cons"].abs()

    consistency_day = merged.groupby("day").apply(
        lambda g: 1.0 - (g["abs_diff"].sum() / (g["abs_ref"].sum() + 1e-9))
    ).reset_index(name="consistency")
    consistency_day["consistency"] = consistency_day["consistency"].clip(0, 1)

    cards = mp_day.groupby("day")[["completeness","validity","plausibility"]].mean().reset_index()
    cards = cards.merge(consistency_day, on="day")

    days = sorted(cards["day"].unique())

    fig = make_subplots(rows=1, cols=4, specs=[[{"type":"indicator"}]*4])
    traces_per_day = 4

    def ind(v, t):
        return go.Indicator(mode="number", value=float(v)*100,
                            number={"suffix":"%","valueformat":".1f"},
                            title={"text":t})

    for d in days:
        r = cards[cards["day"] == d].iloc[0]
        fig.add_trace(ind(r["completeness"], "Completeness"), row=1, col=1)
        fig.add_trace(ind(r["consistency"],   "Consistency"),      row=1, col=2)
        fig.add_trace(ind(r["validity"],      "Validity"),       row=1, col=3)
        fig.add_trace(ind(r["plausibility"],  "Plausibility"),   row=1, col=4)

    for i in range(len(fig.data)):
        fig.data[i].visible = i < traces_per_day

    fig.update_layout(height=280, margin=dict(l=30, r=30, t=60, b=30))

    return fig, cards, {"days":[str(d) for d in days], "traces_per_day":traces_per_day}


In [ ]:
def chapter2_energyflow_multiday(whole_eeg, org_id, gen_col):
    df = whole_eeg.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df[df["org_id"] == org_id]

    df["eigen"] = np.minimum(df[gen_col], df["sum_wt_meas_cons"])
    df["einspeis"] = np.maximum(df[gen_col] - df["sum_wt_meas_cons"], 0)
    df["netz"] = np.maximum(df["sum_wt_meas_cons"] - df[gen_col], 0)

    df["day"] = df["time"].dt.date
    days = sorted(df["day"].unique())

    fig = go.Figure()
    for d in days:
        ddf = df[df["day"] == d]
        fig.add_bar(x=ddf["time"], y=ddf["eigen"], visible=False, name="Eigenverbrauch")
        fig.add_bar(x=ddf["time"], y=ddf["einspeis"], visible=False, name="Einspeisung")
        fig.add_bar(x=ddf["time"], y=ddf["netz"], visible=False, name="Netzbezug")

    for i in range(3):
        fig.data[i].visible = True

    fig.update_layout(barmode="stack",height=520)
    return fig, {"days":[str(d) for d in days], "traces_per_day":3}

In [ ]:
def chapter3_consistency_multiday(all_mps, whole_eeg, org_id, tol=0.01, eps=1e-6):
    mps = all_mps.copy()
    eeg = whole_eeg.copy()
    mps["time"] = pd.to_datetime(mps["time"], utc=True)
    eeg["time"] = pd.to_datetime(eeg["time"], utc=True)

    mps = mps[mps["org_id"] == org_id]
    eeg = eeg[eeg["org_id"] == org_id]

    agg = mps.groupby("time")["wt_meas_cons"].sum().reset_index(name="sum_mps")
    df = agg.merge(eeg[["time","sum_wt_meas_cons"]], on="time", how="inner").sort_values("time")
    df["day"] = df["time"].dt.date

    df["abs_diff"] = (df["sum_mps"] - df["sum_wt_meas_cons"]).abs()
    df["diff"] = np.where(
        df["sum_wt_meas_cons"].abs() < eps,
        0.0,
        df["abs_diff"] / df["sum_wt_meas_cons"].abs() * 100
    )
    df["ok"] = df["diff"] <= tol*100

    days = sorted(df["day"].unique())
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True)

    for d in days:
        ddf = df[df["day"] == d]
        fig.add_trace(go.Scatter(x=ddf["time"], y=ddf["sum_mps"], visible=False, name="Sum MPs"), row=1, col=1)
        fig.add_trace(go.Scatter(x=ddf["time"], y=ddf["sum_wt_meas_cons"], visible=False, name="whole_eeg"), row=1, col=1)

        
        fig.add_trace(go.Scatter(x=ddf["time"], y=ddf["diff"], visible=False, name="Deviation %"), row=2, col=1)

        
        bad = ~ddf["ok"]
        fig.add_trace(go.Scatter(x=ddf["time"][bad], y=ddf["diff"][bad],
                                 visible=False, mode="markers", name="Fail"), row=2, col=1)

    for i in range(4):
        fig.data[i].visible = True

    fig.update_layout(
        title="Chapter 3 – Consistency",
        height=720,
        margin=dict(l=40, r=40, t=80, b=40)
    )
    fig.update_yaxes(title_text="Energy [kWh]", row=1, col=1)
    fig.update_yaxes(title_text="Deviation [%]", row=2, col=1)

    return fig, {"days":[str(d) for d in days], "traces_per_day":4}


In [ ]:
def chapter4_daily_profile_multiday(whole_eeg, org_id, gen_col,
                                    cons_col="sum_wt_meas_cons"):
    df = whole_eeg.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df[df["org_id"] == org_id].sort_values("time")

    if cons_col not in df.columns:
        raise ValueError(f"Missing column: {cons_col}")
    if gen_col not in df.columns:
        raise ValueError(f"Missing column: {gen_col}")

    df["day"] = df["time"].dt.date
    df["slot"] = df["time"].dt.hour * 4 + (df["time"].dt.minute // 15)  # 0..95
    days = sorted(df["day"].unique())
    if not days:
        raise ValueError("No days for this org_id")

    
    x = [(pd.Timestamp("2000-01-01") + pd.Timedelta(minutes=15*i)).strftime("%H:%M") for i in range(96)]

    
    def qprof(col):
        g = df.groupby("slot")[col]
        
        idx = pd.Index(range(96), name="slot")
        p10 = g.quantile(0.10).reindex(idx).to_numpy()
        p50 = g.median().reindex(idx).to_numpy()
        p90 = g.quantile(0.90).reindex(idx).to_numpy()
        return p10, p50, p90

    c10_ref, c50_ref, c90_ref = qprof(cons_col)
    g10_ref, g50_ref, g90_ref = qprof(gen_col)

    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                        subplot_titles=("Consumption", "Generation"))

    
    fig.add_trace(go.Scatter(x=x, y=c90_ref, mode="lines", line=dict(width=0),
                             showlegend=False, hoverinfo="skip"),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=c10_ref, mode="lines", line=dict(width=0),
                             fill="tonexty", name="Consumption band (P10–P90)",
                             hoverinfo="skip"),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=c50_ref, mode="lines", name="Consumption median (overall)",
                             hovertemplate="%{x}<br>Median=%{y:.3f}<extra></extra>"),
                  row=1, col=1)

    
    fig.add_trace(go.Scatter(x=x, y=g90_ref, mode="lines", line=dict(width=0),
                             showlegend=False, hoverinfo="skip"),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=x, y=g10_ref, mode="lines", line=dict(width=0),
                             fill="tonexty", name="Generation band (P10–P90)",
                             hoverinfo="skip"),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=x, y=g50_ref, mode="lines", name="Generation median (overall)",
                             hovertemplate="%{x}<br>Median=%{y:.3f}<extra></extra>"),
                  row=2, col=1)

    
    for d in days:
        ddf = df[df["day"] == d]
       
        c_day = ddf.set_index("slot")[cons_col].reindex(range(96)).to_numpy()
        g_day = ddf.set_index("slot")[gen_col].reindex(range(96)).to_numpy()

        fig.add_trace(go.Scatter(
            x=x, y=c_day, mode="lines", name=f"Consumption (day)",
            visible=False,
            hovertemplate="%{x}<br>Day=%{y:.3f}<extra></extra>"
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=x, y=g_day, mode="lines", name=f"Generation (day)",
            visible=False,
            hovertemplate="%{x}<br>Day=%{y:.3f}<extra></extra>"
        ), row=2, col=1)

    
    base_traces = 6  
    fig.data[base_traces + 0].visible = True
    fig.data[base_traces + 1].visible = True

   
    tick_idx = list(range(0, 96, 8))
    tickvals = [x[i] for i in tick_idx]
    fig.update_xaxes(tickmode="array", tickvals=tickvals, ticktext=tickvals, row=2, col=1)

    fig.update_layout(
        height=650,
        margin=dict(l=40, r=40, t=90, b=40),
        legend_title="Series"
    )

    meta = {"days": [str(d) for d in days], "traces_per_day": 2, "base_traces": base_traces}
    return fig, meta


In [ ]:
def chapter6_go_nogo_period(cards,
                            green=0.90, yellow=0.85,
                            cons_col="consistency",
                            w=(0.30, 0.30, 0.20, 0.20),
                            agg="mean"):
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    c = cards.copy()
    c["day"] = pd.to_datetime(c["day"]).dt.date
    c = c.sort_values("day")

    need = ["day", "completeness", cons_col, "validity", "plausibility"]
    miss = [x for x in need if x not in c.columns]
    if miss:
        raise ValueError(f"cards missing columns: {miss}")

    W = np.array(w, dtype=float)
    W = W / W.sum()

    X = c[["completeness", cons_col, "validity", "plausibility"]].to_numpy(float)
    c["score_day"] = np.clip((X * W).sum(axis=1), 0, 1)

    score = float(np.median(c["score_day"])) if agg == "median" else float(np.mean(c["score_day"]))

    def label(s):
        if s >= green:
            return "GO", "green"
        if s >= yellow:
            return "CONDITIONAL GO", "orange"
        return "NO-GO", "red"

    lab, col = label(score)

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=score * 100,
        number={"suffix": "%", "valueformat": ".1f"},
        title={"text": f"Overall data quality assessment: {lab}"},
        gauge={
            "axis": {"range": [0, 100]},
            "bar": {"color": col},
            "steps": [
                {"range": [0, yellow * 100], "color": "#f4cccc"},
                {"range": [yellow * 100, green * 100], "color": "#fff2cc"},
                {"range": [green * 100, 100], "color": "#d9ead3"},
            ],
            "threshold": {"line": {"width": 3}, "thickness": 0.8, "value": green * 100}
        }
    ))

    fig.update_layout(height=360, margin=dict(l=30, r=30, t=80, b=30))
    return fig, c


In [ ]:
import json
import plotly.offline as po

def plotly_js_inline():
    return po.get_plotlyjs()

def export_html(path, sections, controller, captions):
    days = sorted(list(set.intersection(*[set(v["days"]) for v in controller.values()])))
    if not days:
        raise ValueError("No common day found between the controlled plots (controller).")

    html = ["""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Data Quality Handbook</title>
  <style>
    body{font-family:Arial;margin:24px}
    .toolbar{margin:12px 0 18px 0}
    .section{margin-top:30px}
    .caption{color:#444;margin-top:6px}
  </style>
  <script>""" + plotly_js_inline() + """</script>
</head>
<body>
  <h1>Data Quality Handbook</h1>
  <div class="toolbar">
    <label><b>Day:</b></label>
    <select id="day"></select>
  </div>
"""]

    
    for s in sections:
        div_id = s["div_id"]
        html.append(f"<div class='section'><h2>{s['title']}</h2>")
        html.append(s["fig"].to_html(include_plotlyjs=False, full_html=False, div_id=div_id))
        if div_id in captions:
            html.append(f"<div class='caption'>{captions[div_id]}</div>")
        html.append("</div>")

    
    controller_json = json.dumps(controller)
    days_json = json.dumps(days)

    script = """
<script>
const controller = __CONTROLLER__;
const days = __DAYS__;
const sel = document.getElementById("day");

days.forEach(d => {
  const o = document.createElement("option");
  o.value = d; o.text = d;
  sel.appendChild(o);
});

function setDay(d) {
  for (const [id, m] of Object.entries(controller)) {
    const i = m.days.indexOf(d);
    if (i < 0) continue;

    const base = m.base_traces || 0;
    const total = base + (m.days.length * m.traces_per_day);

    const vis = Array(total).fill(true);
    for (let j = base; j < total; j++) vis[j] = false;

    const start = base + i * m.traces_per_day;
    for (let k = 0; k < m.traces_per_day; k++) vis[start + k] = true;

    Plotly.restyle(id, { "visible": vis });
  }
}

sel.onchange = e => setDay(e.target.value);
sel.value = days[0];
setDay(days[0]);
</script>
</body>
</html>
""".replace("__CONTROLLER__", controller_json).replace("__DAYS__", days_json)

    html.append(script)

    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(html))


In [ ]:
ORG_ID = 1
GEN_COL = "sum_wt_meas_gen"

fig1, cards, meta1 = chapter1_kpis(all_mps, whole_eeg, ORG_ID)
fig2, meta2 = chapter2_energyflow_multiday(whole_eeg, ORG_ID, GEN_COL)
fig3, meta3 = chapter3_consistency_multiday(all_mps, whole_eeg, ORG_ID)
fig4, meta4 = chapter4_daily_profile_multiday(whole_eeg, ORG_ID, GEN_COL)
fig6, cards_scored = chapter6_go_nogo_period(cards, cons_col="consistency")



sections = [
    {"title":"Chapter 1 – KPI & MP score", "fig":fig1, "div_id":"fig1"},
    {"title":"Chapter 2 – Energy flow", "fig":fig2, "div_id":"fig2"},
    {"title":"Chapter 3 – Consistency", "fig":fig3, "div_id":"fig3"},
    {"title":"Chapter 4 – Daily profile", "fig":fig4, "div_id":"fig4"},
    {"title":"Chapter 5 – Go / No-Go", "fig":fig6, "div_id":"fig6"},
]

captions = {
    "fig1":"This chart shows key data-quality indicators on a daily basis. The completeness, validity, and plausibility values are computed from the 15-minute measurement data of all metering points, while consistency reflects the reconciliation between aggregated individual measurements and the organisation totals. The visualisation allows a quick assessment of the stability and reliability of the underlying data.",
    "fig2":"This chart shows the energy flow within the energy community in 15-minute intervals. It is based on the measured consumption and generation values, from which self-consumption, feed-in, and grid draw are derived. The visualisation gives an intuitive understanding of the temporal distribution and composition of the energy flows.",
    "fig3":"This chart compares the aggregated sum of all metering-point measurements with the corresponding organisation totals over time. The percentage deviation between the two quantities is also shown, to make inconsistencies visible. The analysis is based on a direct interval-to-interval comparison of the measurement data.",
    "fig4":"This chart shows typical daily patterns of consumption and generation, aggregated over multiple days. It shows the median as well as a range between low and high quantiles, computed from the 15-minute time series. This makes recurring patterns and deviations from normal behaviour identifiable.",
    "fig6":"This chart shows an aggregated overall data-quality assessment for the entire observation period. First, a score is computed for each day by combining the completeness, consistency, validity, and plausibility indicators with weights. The final score is the mean of these daily scores and is classified as GO, CONDITIONAL GO, or NO-GO based on defined thresholds."
}

controller = {
    "fig1": meta1,
    "fig2": meta2,
    "fig4": meta4,
    "fig3": meta3,
    
}

export_html("dq_handbook.html", sections, controller, captions)
print("dq_handbook.html created")